### 1.1 Define the Business Process and the Fact Grain

**Business process:**
The business process analyzed in this project is the recording of financial transactions from an investment account during 2024. Each transaction represents either a BUY or SELL operation of a traded stock.

**Fact grain:**
The grain of the fact table is one individual transaction. Therefore, one row in `Fact_Transactions` represents one transaction from the original account statement file.

### 1.2 Identify Fact and Dimensions

The star schema consists of one central fact table and four dimension tables.

| Type | Table |
|---|---|
| Fact table | `Fact_Transactions` |
| Dimension table | `Dim_Time` |
| Dimension table | `Dim_Geography` |
| Dimension table | `Dim_Symbol` |
| Dimension table | `Dim_TransactionType` |

`Fact_Transactions` stores the measurable transactional data and foreign keys to all dimensions.
The dimension tables store descriptive attributes used for filtering, grouping, and analysis.

### 1.3 Define Dimension Hierarchies

The following hierarchies are used in the dimensional model:

**Dim_Time:**
Day → Month → Quarter → Year

**Dim_Geography:**
Region → Sub-region → Country

**Dim_Symbol:**
Company → Industry → Sector

**Dim_TransactionType:**
This dimension is flat because transaction type only distinguishes between BUY and SELL.

### 1.4 Design the Star Schema

The final star schema contains one central fact table and four surrounding dimension tables.

**Fact_Transactions**
- `transaction_key`
- `time_key`
- `symbol_key`
- `geo_key`
- `type_key`
- `Unit`

**Dim_Time**
- `time_key`
- `Date`
- `Day`
- `Month`
- `Quarter`
- `Year`
- `Weekday`

**Dim_Geography**
- `geo_key`
- `country`
- `alpha_3`
- `region`
- `sub_region`

**Dim_Symbol**
- `symbol_key`
- `Symbol`
- `company_name`
- `sector`
- `industry`
- `country`

**Dim_TransactionType**
- `type_key`
- `TransactionType`

The fact table stores only foreign keys and the numerical measure `Unit`. Descriptive attributes such as company name, sector, industry, country, region, and date details are stored in the corresponding dimension tables. This reduces redundancy and makes analytical SQL queries easier to write.

### Star Schema Summary

The star schema organizes the transactional data into one central fact table and several dimension tables. This structure separates numerical measures from descriptive information, reducing data redundancy and making the dataset easier to maintain. It also simplifies analytical queries by allowing dimensions such as time, geography, company, and transaction type to be joined efficiently with the fact table. Overall, the model provides a scalable foundation for business intelligence tasks, SQL analysis, and interactive dashboard development.

In [108]:
import pandas as pd

fact_raw = pd.read_csv("account-statement-1-1-2024-12-31-2024.csv", sep=";")
dim_symbol_raw = pd.read_csv("symbols.csv", sep=";")
dim_geography_raw = pd.read_csv("country.csv")

In [109]:
fact_raw.columns = fact_raw.columns.str.strip()
dim_symbol_raw.columns = dim_symbol_raw.columns.str.strip()
dim_geography_raw.columns = dim_geography_raw.columns.str.strip()

dim_symbol_raw = dim_symbol_raw.rename(columns={"symbol": "Symbol"})

fact_raw["Date"] = pd.to_datetime(
    fact_raw["Date"],
    dayfirst=True,
    errors="coerce"
)

In [110]:
fact_raw = fact_raw.dropna(
    subset=["IDTransaction"]
).reset_index(drop=True)

fact_raw = fact_raw.drop(columns=["Unnamed: 5"])
fact_raw.isna().sum()

IDTransaction      0
Date               0
TransactionType    0
Symbol             0
Unit               0
dtype: int64

In [112]:
dim_geography = (
    dim_geography_raw[
        ["name", "alpha-3", "region", "sub-region"]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

dim_geography = dim_geography.rename(columns={
    "name": "country",
    "alpha-3": "alpha_3",
    "sub-region": "sub_region"
})

dim_geography["geo_key"] = dim_geography.index + 1

print(dim_geography)

               country alpha_3   region          sub_region  geo_key
0          Afghanistan     AFG     Asia       Southern Asia        1
1        Åland Islands     ALA   Europe     Northern Europe        2
2              Albania     ALB   Europe     Southern Europe        3
3              Algeria     DZA   Africa     Northern Africa        4
4       American Samoa     ASM  Oceania           Polynesia        5
..                 ...     ...      ...                 ...      ...
244  Wallis and Futuna     WLF  Oceania           Polynesia      245
245     Western Sahara     ESH   Africa     Northern Africa      246
246              Yemen     YEM     Asia        Western Asia      247
247             Zambia     ZMB   Africa  Sub-Saharan Africa      248
248           Zimbabwe     ZWE   Africa  Sub-Saharan Africa      249

[249 rows x 5 columns]


In [113]:
dim_symbol = (
    dim_symbol_raw[
        ["Symbol", "company_name", "sector", "industry", "country"]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

dim_symbol["symbol_key"] = dim_symbol.index + 1
print(dim_symbol)

     Symbol                     company_name              sector  \
0      TEAM            Atlassian Corporation          Technology   
1       WDS    Woodside Energy Group Limited              Energy   
2       OSW     OneSpaWorld Holdings Limited   Consumer Cyclical   
3      ACGL          Arch Capital Group Ltd.  Financial Services   
4       AGO            Assured Guaranty Ltd.  Financial Services   
...     ...                              ...                 ...   
3189   ZURA                 Zura Bio Limited          Healthcare   
3190   ZVRA         Zevra Therapeutics, Inc.          Healthcare   
3191    ZWS  Zurn Elkay Water Solutions Corp         Industrials   
3192   ZYME                   Zymeworks Inc.          Healthcare   
3193   ZYXI                      Zynex, Inc.          Healthcare   

                            industry                   country  symbol_key  
0             Software - Application                 Australia           1  
1                      Oil & 

In [114]:
dim_time = (
    fact_raw[["Date"]]
    .drop_duplicates()
    .sort_values("Date")
    .reset_index(drop=True)
)

dim_time["time_key"] = dim_time.index + 1
dim_time["day"] = dim_time["Date"].dt.day
dim_time["month"] = dim_time["Date"].dt.month
dim_time["quarter"] = dim_time["Date"].dt.quarter
dim_time["year"] = dim_time["Date"].dt.year

print(dim_time)

                    Date  time_key  day  month  quarter  year
0    2024-01-02 14:33:01         1    2      1        1  2024
1    2024-01-02 14:38:20         2    2      1        1  2024
2    2024-01-02 14:42:43         3    2      1        1  2024
3    2024-01-02 19:17:25         4    2      1        1  2024
4    2024-01-03 08:00:11         5    3      1        1  2024
...                  ...       ...  ...    ...      ...   ...
2208 2024-12-27 02:59:57      2209   27     12        4  2024
2209 2024-12-27 15:05:10      2210   27     12        4  2024
2210 2024-12-27 15:37:02      2211   27     12        4  2024
2211 2024-12-30 14:30:09      2212   30     12        4  2024
2212 2024-12-30 15:00:09      2213   30     12        4  2024

[2213 rows x 6 columns]


In [115]:
dim_transaction = (
    fact_raw[["TransactionType"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

dim_transaction["type_key"] = dim_transaction.index + 1
print(dim_transaction)

  TransactionType  type_key
0             BUY         1
1            SELL         2
2        DIVIDENT         3


In [116]:
missing_countries = (
    set(dim_symbol["country"].dropna())
    - set(dim_geography["country"].dropna())
)

print(missing_countries)

{'Turkey', 'Taiwan'}


In [117]:
country_corrections = {
    "Turkey": "Türkiye",
    "Taiwan": "Taiwan, Province of China"
}

dim_symbol["country"] = dim_symbol["country"].replace(country_corrections)

missing_countries = (
    set(dim_symbol["country"].dropna())
    - set(dim_geography["country"].dropna())
)

print(missing_countries)

set()


Two country names were standardized before mapping: Turkey was changed to Türkiye and Taiwan was changed to Taiwan, Province of China. After this correction, all company countries could be mapped to the geography dimension.

In [137]:
dim_symbol = dim_symbol[
    ["symbol_key", "Symbol", "company_name", "sector", "industry", "country"]
].copy()

dim_symbol = dim_symbol.merge(
    dim_geography[["geo_key", "country"]],
    on="country",
    how="left"
)


fact_transactions = fact_raw.merge(
    dim_time[["time_key", "Date"]],
    on="Date",
    how="left"
)

fact_transactions = fact_transactions.merge(
    dim_symbol[["symbol_key", "geo_key", "Symbol"]],
    on="Symbol",
    how="left"
)

fact_transactions = fact_transactions.merge(
    dim_transaction[["type_key", "TransactionType"]],
    on="TransactionType",
    how="left"
)

fact_transactions = fact_transactions[
    ["IDTransaction", "time_key", "symbol_key", "geo_key", "type_key", "Unit"]
]

fact_transactions.head()

,IDTransaction,time_key,symbol_key,geo_key,type_key,Unit
0,2.769834e+09,78,284.0,175.0,1,1605.0
1,2.767325e+09,154,284.0,175.0,2,1605.0
2,2.815474e+09,67,284.0,175.0,2,914.0
3,2.622244e+09,118,4.0,25.0,1,646.0
4,2.629871e+09,119,258.0,131.0,2,646.0


In [138]:
fact_transactions.isna().sum()

IDTransaction      0
time_key           0
symbol_key       212
geo_key          212
type_key           0
Unit               0
dtype: int64

In [140]:
fact_transactions = fact_transactions.dropna().reset_index(drop=True)
fact_transactions.isna().sum()


IDTransaction    0
time_key         0
symbol_key       0
geo_key          0
type_key         0
Unit             0
dtype: int64

Transactions with missing dimension mappings were removed from the final fact table because they could not be linked to the symbol and geography dimensions. After this cleaning step, the final fact table contains only complete records with valid foreign keys and no missing values.

In [144]:
import sqlite3

conn = sqlite3.connect(':memory:')

fact_transactions.to_sql('fact_transactions', conn, index=False)
dim_symbol.to_sql('dim_symbol', conn, index=False)
dim_transaction.to_sql('dim_transaction', conn, index=False)
dim_geography.to_sql('dim_geography', conn, index=False)
dim_time.to_sql('dim_time', conn, index=False)

ot1 = """
SELECT
    ds.sector,
    COUNT(*) AS sell_transaction_count
FROM fact_transactions ft
JOIN dim_symbol ds
    ON ft.symbol_key = ds.symbol_key
JOIN dim_transaction dt
    ON ft.type_key = dt.type_key
JOIN dim_geography dg
    ON ft.geo_key = dg.geo_key
JOIN dim_time tm
    ON ft.time_key = tm.time_key
WHERE dt.TransactionType = 'SELL'
  AND dg.country = 'United States of America'
  AND tm.year = 2024
GROUP BY ds.sector
ORDER BY sell_transaction_count DESC
LIMIT 5;
"""

result_ot1 = pd.read_sql_query(ot1, conn)
result_ot1

,sector,sell_transaction_count
0,Technology,158
1,Communication Services,58
2,Financial Services,55
3,Healthcare,50
4,Consumer Cyclical,48


In [145]:
ot2 = """
SELECT
    ds.industry,
    COUNT(*) AS buy_transaction_count
FROM fact_transactions ft
JOIN dim_symbol ds
    ON ft.symbol_key = ds.symbol_key
JOIN dim_transaction dt
    ON ft.type_key = dt.type_key
JOIN dim_time tm
    ON ft.time_key = tm.time_key
WHERE dt.TransactionType = 'BUY'
  AND tm.year = 2024
  AND tm.quarter = 4
GROUP BY ds.industry
ORDER BY buy_transaction_count DESC
LIMIT 5;
"""

result_ot2 = pd.read_sql_query(ot2, conn)
result_ot2

,industry,buy_transaction_count
0,Semiconductors,18
1,Internet Content & Information,15
2,Software - Infrastructure,10
3,Internet Retail,8
4,Diagnostics & Research,7


In [146]:
ot3 = """
SELECT
    tm.quarter,
    COUNT(*) AS total_transaction_count
FROM fact_transactions ft
JOIN dim_time tm
    ON ft.time_key = tm.time_key
JOIN dim_transaction dt
    ON ft.type_key = dt.type_key
WHERE tm.year = 2024
  AND dt.TransactionType IN ('BUY', 'SELL')
GROUP BY tm.quarter
ORDER BY total_transaction_count DESC;
"""

result_ot3 = pd.read_sql_query(ot3, conn)
result_ot3

,quarter,total_transaction_count
0,1,968
1,2,522
2,3,242
3,4,241


In [147]:
ot4 = """
SELECT
    dg.country,
    COUNT(*) AS sell_transaction_count
FROM fact_transactions ft
JOIN dim_geography dg
    ON ft.geo_key = dg.geo_key
JOIN dim_transaction dt
    ON ft.type_key = dt.type_key
JOIN dim_time tm
    ON ft.time_key = tm.time_key
WHERE dt.TransactionType = 'SELL'
  AND tm.year = 2024
GROUP BY dg.country
ORDER BY sell_transaction_count DESC
LIMIT 10;
"""

result_ot4 = pd.read_sql_query(ot4, conn)
result_ot4

,country,sell_transaction_count
0,United States of America,389
1,United Kingdom of Great Britain and Northern I...,130
2,China,112
3,Brazil,69
4,"Taiwan, Province of China",50
5,"Netherlands, Kingdom of the",46
6,Switzerland,37
7,Ireland,31
8,Luxembourg,27
9,Canada,22


In [148]:
ot5 = """
SELECT
    dg.region,
    SUM(ft.Unit) AS total_units_bought
FROM fact_transactions ft
JOIN dim_geography dg
    ON ft.geo_key = dg.geo_key
JOIN dim_transaction dt
    ON ft.type_key = dt.type_key
JOIN dim_time tm
    ON ft.time_key = tm.time_key
WHERE dt.TransactionType = 'BUY'
  AND tm.year = 2024
GROUP BY dg.region
ORDER BY total_units_bought DESC
LIMIT 5;
"""

result_ot5 = pd.read_sql_query(ot5, conn)
result_ot5

,region,total_units_bought
0,Americas,37026.0
1,Europe,22528.0
2,Asia,9198.0
3,None,2339.0


In [150]:
ot6 = """
SELECT
    ds.sector,
    SUM(ft.Unit) AS total_units_traded
FROM fact_transactions ft
JOIN dim_symbol ds
    ON ft.symbol_key = ds.symbol_key
JOIN dim_time tm
    ON ft.time_key = tm.time_key
WHERE tm.year = 2024
  AND tm.quarter = 3
GROUP BY ds.sector
ORDER BY total_units_traded DESC
LIMIT 5;
"""

result_ot6 = pd.read_sql_query(ot6, conn)
result_ot6

,sector,total_units_traded
0,Technology,4721.0
1,Healthcare,3771.0
2,Consumer Cyclical,2667.0
3,Financial Services,2358.0
4,Communication Services,2116.0


In [151]:
ot7 = """
SELECT
    ds.Symbol,
    COUNT(*) AS transaction_count
FROM fact_transactions ft
JOIN dim_symbol ds
    ON ft.symbol_key = ds.symbol_key
JOIN dim_time tm
    ON ft.time_key = tm.time_key
WHERE tm.year = 2024
GROUP BY ds.Symbol
ORDER BY transaction_count DESC
LIMIT 10;
"""

result_ot7 = pd.read_sql_query(ot7, conn)
result_ot7

,Symbol,transaction_count
0,ARM,100
1,AMD,97
2,TSM,80
3,TIMB,76
4,GOOG,52
5,MSFT,49
6,AMZN,47
7,ARDX,43
8,BRFS,42
9,BLK,42


In [152]:
ot8 = """
SELECT
    dg.country,
    SUM(ft.Unit) AS total_units_traded
FROM fact_transactions ft
JOIN dim_symbol ds
    ON ft.symbol_key = ds.symbol_key
JOIN dim_geography dg
    ON ft.geo_key = dg.geo_key
WHERE ds.sector = 'Financial Services'
GROUP BY dg.country
ORDER BY total_units_traded DESC
LIMIT 5;
"""

result_ot8 = pd.read_sql_query(ot8, conn)
result_ot8

,country,total_units_traded
0,United States of America,7352.0
1,Peru,4434.0
2,Bermuda,3658.0
3,China,1663.0
4,Canada,1466.0


Question 1

The Technology sector had by far the highest number of SELL transactions in the United States during 2024, with a significant lead over the remaining sectors. Communication Services and Financial Services ranked second and third.

The query joins the fact table with the symbol, transaction, geography, and time dimensions. It filters SELL transactions from the United States in 2024, groups the results by sector, and returns the five sectors with the highest number of sell transactions.

Question 2

In Q4 2024, the Semiconductor industry recorded the highest number of BUY transactions. Internet Content & Information and Software – Infrastructure were also among the most active industries.

The query joins the fact table with the symbol and time dimensions. It filters BUY transactions from the fourth quarter of 2024, groups the data by industry, and counts the number of transactions for each industry.

Question 3

The first quarter (Q1) had the highest transaction activity in 2024, followed by Q2. Transaction volumes declined considerably in Q3 and Q4.

The query joins the fact table with the time and transaction dimensions. It selects all BUY and SELL transactions from 2024, groups them by quarter, and counts the total number of transactions in each quarter.

Question 4

The United States of America recorded the highest number of SELL transactions in 2024. The United Kingdom and China followed in second and third place, while the remaining countries had noticeably lower transaction counts.

The query joins the fact table with the geography, transaction, and time dimensions. It filters SELL transactions in 2024, groups the results by country, and returns the countries with the highest number of sell transactions.

Question 5

The Americas had the highest total number of units bought in 2024, followed by Europe. Asia ranked third, while transactions without a defined region represented only a small share.

The query joins the fact table with the geography, transaction, and time dimensions. It filters BUY transactions from 2024, sums the traded units for each region, and returns the regions with the highest trading volume.

Question 6

During Q3 2024, the Technology sector recorded the highest number of traded units. Healthcare and Consumer Cyclical also showed strong trading activity, while the remaining sectors traded noticeably lower volumes.

The query joins the fact table with the symbol and time dimensions. It filters transactions from the third quarter of 2024, groups them by sector, and calculates the total number of traded units for each sector.

Question 7

ARM was the most actively traded symbol in 2024, closely followed by AMD and TSM. These companies clearly dominated transaction activity compared to the rest of the top ten.

The query joins the fact table with the symbol and time dimensions. It selects all transactions from 2024, groups them by stock symbol, and counts the number of transactions for each symbol.

Question 8

Within the Financial Services sector, the United States accounted for the highest number of traded units. Peru and Bermuda followed with substantial trading volumes, while the remaining countries contributed significantly fewer units.

The query joins the fact table with the symbol and geography dimensions. It filters transactions belonging to the Financial Services sector, groups them by country, and calculates the total traded units for each country.

BRIEF REPORT:

The dataset was transformed into a star schema consisting of one fact table and four dimension tables (Time, Symbol, Geography, and Transaction). This structure simplified the analytical process and enabled efficient SQL queries across multiple business dimensions. During data preparation, several quality issues were addressed, including inconsistent date formats, unnecessary spaces in column names, and the integration of multiple datasets using common keys. The cleaned data was then loaded into the star schema and analyzed using SQL.

The analysis showed that the Technology sector dominated SELL transactions in the United States during 2024, while the Semiconductor industry recorded the highest number of BUY transactions in the fourth quarter. Transaction activity was highest in the first quarter of the year and gradually decreased during the remaining quarters. The United States had by far the largest number of SELL transactions compared to other countries. The Americas also recorded the highest total volume of purchased units among all regions. Technology was the leading sector by traded volume in the third quarter, and symbols such as ARM, AMD, and TSM appeared among the most frequently traded assets. Overall, the star schema enabled efficient multidimensional analysis and provided meaningful insights into transaction patterns across industries, countries, regions, and time periods.
